# Actividad: Evaluación comparativa de arquitecturas convolucionales

En este notebook se construyen, entrenan y comparan tres modelos para clasificación en CIFAR-10:
1. CNN propia simple
2. CNN propia profunda
3. Arquitectura clásica con transfer learning (MobileNetV2)

Además, se incluye data augmentation, regularización y una comparación experimental final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2

print('TensorFlow:', tf.__version__)
SEED = 42
tf.keras.utils.set_random_seed(SEED)

## Carga y preparación de datos

In [ ]:
# CIFAR-10: 60,000 imágenes de 32x32x3 en 10 clases
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Split de validación desde entrenamiento
val_size = 5000
x_val, y_val = x_train[-val_size:], y_train[-val_size:]
x_train, y_train = x_train[:-val_size], y_train[:-val_size]

# Convertir a float32 y normalizar a [0,1]
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = y_train.squeeze()
y_val = y_val.squeeze()
y_test = y_test.squeeze()

print('Train:', x_train.shape, y_train.shape)
print('Val  :', x_val.shape, y_val.shape)
print('Test :', x_test.shape, y_test.shape)

In [ ]:
# Visualización rápida
plt.figure(figsize=(10, 4))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## Data augmentation y utilidades

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name='data_augmentation')

def plot_history(history, title='Model history'):
    hist = history.history
    epochs = range(1, len(hist['loss']) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist['loss'], label='train_loss')
    plt.plot(epochs, hist['val_loss'], label='val_loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, hist['accuracy'], label='train_acc')
    plt.plot(epochs, hist['val_accuracy'], label='val_acc')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

## Definiciones de modelos

In [ ]:
def build_cnn_simple(input_shape=(32, 32, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='cnn_simple')

def build_cnn_deep(input_shape=(32, 32, 3), num_classes=10):
    l2 = regularizers.l2(1e-4)
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Bloque 1
    x = layers.Conv2D(32, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    # Bloque 2
    x = layers.Conv2D(64, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    # Bloque 3
    x = layers.Conv2D(128, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.40)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2)(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='cnn_deep')

def build_transfer_model(input_shape=(32, 32, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # MobileNetV2 fue preentrenado con imágenes mayores, por eso hacemos resize
    x = layers.Resizing(96, 96)(x)

    base_model = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(96, 96, 3)
    )
    base_model.trainable = False

    x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='mobilenetv2_transfer')
    return model, base_model

In [ ]:
cnn_simple = build_cnn_simple()
cnn_deep = build_cnn_deep()
transfer_model, transfer_base = build_transfer_model()

cnn_simple.summary()

## Entrenamiento de modelos

In [ ]:
BATCH_SIZE = 128
EPOCHS_SCRATCH = 18
EPOCHS_TRANSFER_HEAD = 10
EPOCHS_TRANSFER_FINETUNE = 6

callbacks_common = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
]

def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

In [ ]:
# 1) Entrenamiento CNN simple
compile_model(cnn_simple, lr=1e-3)
history_simple = cnn_simple.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_SCRATCH,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_common,
    verbose=1
)

In [ ]:
# 2) Entrenamiento CNN profunda
compile_model(cnn_deep, lr=1e-3)
history_deep = cnn_deep.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_SCRATCH,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_common,
    verbose=1
)

In [ ]:
# 3) Transfer learning (fase 1: solo cabezal)
compile_model(transfer_model, lr=1e-3)
history_transfer_head = transfer_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_TRANSFER_HEAD,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_common,
    verbose=1
)

In [ ]:
# Fine-tuning: descongelar últimas capas del backbone
transfer_base.trainable = True

for layer in transfer_base.layers[:-30]:
    layer.trainable = False

compile_model(transfer_model, lr=1e-5)
history_transfer_ft = transfer_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_TRANSFER_FINETUNE,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_common,
    verbose=1
)

## Estadística y gráficos

In [ ]:
plot_history(history_simple, title='CNN Simple')
plot_history(history_deep, title='CNN Profunda')
plot_history(history_transfer_head, title='Transfer Learning - Head')
plot_history(history_transfer_ft, title='Transfer Learning - Fine-tuning')

In [ ]:
# Evaluación en test
results = []

for model in [cnn_simple, cnn_deep, transfer_model]:
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results.append({'model': model.name, 'test_loss': loss, 'test_accuracy': acc})

results_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results_df['model'], results_df['test_accuracy'])
plt.title('Comparación de accuracy en test')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
for i, val in enumerate(results_df['test_accuracy']):
    plt.text(i, val + 0.01, f'{val:.3f}', ha='center')
plt.show()

In [ ]:
# Predicciones ejemplo del mejor modelo
best_model_name = results_df.loc[0, 'model']
best_model = {'cnn_simple': cnn_simple, 'cnn_deep': cnn_deep, 'mobilenetv2_transfer': transfer_model}[best_model_name]

pred_probs = best_model.predict(x_test[:16], verbose=0)
preds = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_test[i])
    real = class_names[y_test[i]]
    pred = class_names[preds[i]]
    color = 'green' if preds[i] == y_test[i] else 'red'
    plt.title(f'R:{real}\nP:{pred}', color=color, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

# Conclusiones

En esta práctica se compararon dos arquitecturas CNN propias y una arquitectura clásica con transfer learning.

Con base en el accuracy de test, el modelo con mejor desempeño fue el que aparece en la primera fila de `results_df`. En general, el modelo profundo mejora respecto al modelo simple por su mayor capacidad de extracción de características, pero también requiere más regularización para evitar sobreajuste.

El enfoque de transfer learning suele ofrecer mejor relación desempeño/tiempo de entrenamiento, ya que parte de representaciones preentrenadas en ImageNet. El fine-tuning final ayuda a adaptar esas representaciones al dominio CIFAR-10.

¿Qué mejoraría?
1. Aumentar búsqueda de hiperparámetros (learning rate, dropout, batch size).
2. Probar augmentations adicionales (Cutout, MixUp o RandAugment).
3. Entrenar más épocas con políticas de LR más robustas (cosine decay).
4. Evaluar otras bases preentrenadas (ResNet50, EfficientNet).

Estas mejoras pueden incrementar la robustez y generalización del modelo final.